# Task 5: LLM-as-a-Judge for Product Description Evaluation

This notebook implements an automated judge using a second LLM to evaluate
product descriptions generated in Task 2. The judge scores each description
on five text-quality criteria (Fluency, Grammar, Tone, Length, Grounding)
using the same rubric that was applied during human evaluation in Task 3.

Task 6 then runs the judge on all products, compares its verdicts to human
scores, and explores whether evaluating one criterion at a time improves
agreement with human judgments.

## 5.1 Setup

In [ ]:
import os
import time
import json
from typing import Literal

import pandas as pd
from pydantic import BaseModel
from openai import OpenAI
from dotenv import load_dotenv

# ── Load environment variables ──────────────────────────────────────────
load_dotenv("../.env")

# ── Create OpenAI-compatible client for Nebius AI Studio ────────────────
client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/",
    api_key=os.environ["NEBIUS_API_KEY"],
)

# ── Load human-evaluated Gemma baseline ─────────────────────────────────
df = pd.read_excel("task_03_gemma.xlsx", engine="openpyxl")
print(f"Loaded {len(df)} products from task_03_gemma.xlsx")
print(f"Columns: {df.columns.tolist()}")
print(f"Products with human eval: {df['fluency'].notna().sum()}")

## 5.2 Configuration

Change `JUDGE_MODEL` below to swap the judge model without touching any other code.

In [ ]:
# ── Judge model (the model NOT used for generation in Task 2) ───────────
# Task 2 used google/gemma-2-9b-it-fast, so the default judge is Llama.
# Switch to e.g. "Qwen/Qwen3-30B-A3B-Instruct-2507" if needed.
JUDGE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"

print(f"Judge model: {JUDGE_MODEL}")

## 5.3 Why "Explanation Before Verdict" Matters

In the Pydantic schema below, every criterion result has the `explanation`
field **before** the `verdict` field. This is not arbitrary -- it is a
deliberate design choice that mirrors chain-of-thought prompting.

When a language model generates structured JSON, it produces tokens
left-to-right. If `verdict` came first, the model would commit to a rating
("good", "ok", or "bad") before articulating its reasoning, which encourages
snap judgments and makes the explanation a post-hoc rationalization.

By placing `explanation` first, the model is forced to lay out the evidence
and reasoning *before* it ever emits the verdict token. This:

1. **Reduces hasty errors** -- the model considers the rubric in detail
   before locking in a score.
2. **Improves consistency** -- the explanation grounds the verdict, so
   similar descriptions tend to receive similar ratings.
3. **Aids debugging** -- we can read the explanation to understand *why*
   the judge chose a particular rating and catch systematic biases.

## 5.4 Pydantic Schemas

In [ ]:
# ── Schema for all-criteria-at-once judging ─────────────────────────────

class CriterionResult(BaseModel):
    """A single criterion evaluation. Explanation MUST come before verdict."""
    explanation: str   # reasoning first — forces chain-of-thought
    verdict: Literal["good", "ok", "bad"]


class JudgeOutput(BaseModel):
    """Complete judge output for all five text-quality criteria."""
    fluency: CriterionResult
    grammar: CriterionResult
    tone: CriterionResult
    length: CriterionResult
    grounding: CriterionResult


# ── Schema for single-criterion judging ─────────────────────────────────

class SingleCriterionResult(BaseModel):
    """Judge output when evaluating a single criterion in isolation."""
    explanation: str   # reasoning first
    verdict: Literal["good", "ok", "bad"]


# Quick sanity check: print the JSON schema the API will receive
print("JudgeOutput schema (excerpt):")
print(json.dumps(JudgeOutput.model_json_schema(), indent=2)[:600], "...")

## 5.5 Judge Prompts

The system prompt embeds the full rubric so the judge has explicit criteria.
The user message supplies the source product data and generated description.

In [ ]:
# ── System prompt for ALL-CRITERIA-AT-ONCE judging ──────────────────────

JUDGE_SYSTEM_PROMPT = """\
You are an expert product-copy evaluator. You will receive:
1. SOURCE DATA — the original product information (name, attributes, material, warranty).
2. GENERATED DESCRIPTION — a marketing description produced by an LLM.

Evaluate the description on EXACTLY five criteria using the rubric below.
For EACH criterion, first write a concise explanation of your reasoning,
then give your verdict as one of: "good", "ok", or "bad".

═══ RUBRIC ═══

FLUENCY
  good (3 pts): Smooth, varied sentence structure; flows naturally; no robotic
    repetition; no two sentences start with the same word.
  ok   (2 pts): 1 instance of awkward phrasing, a repeated sentence opener,
    or a robotic transition.
  bad  (1 pt):  3+ awkward phrases, highly repetitive sentence structures,
    or bullet-point style output.

GRAMMAR
  good (3 pts): 0 spelling, punctuation, or grammatical errors.
  ok   (2 pts): 1 minor error (e.g., a missing comma) that does not obscure meaning.
  bad  (1 pt):  2+ errors, or any error that makes a sentence difficult to understand.

TONE
  good (3 pts): Actively persuasive — uses at least two distinct persuasive
    techniques (e.g., benefit framing, emotional/aspirational appeal, sensory
    language, social proof, urgency).
  ok   (2 pts): Uses sales-oriented language but relies on only one persuasive
    technique or describes features without evoking benefit or aspiration
    (informative but flat).
  bad  (1 pt):  Dry/robotic/technical-manual style, overly aggressive, or
    inappropriate.

LENGTH
  good (3 pts): Exactly 55–85 words.
  ok   (2 pts): 50–54 or 86–90 words.
  bad  (1 pt):  Outside the range (<50 or >90 words).

GROUNDING
  good (3 pts): Mentions Name, Material, Warranty, AND all structured
    attributes — woven naturally into sentences, with zero hallucinations or
    invented specs.
  ok   (2 pts): Mentions Name, Material, and Warranty but omits one or more
    structured attributes; OR includes 1 minor plausible but unstated detail.
  bad  (1 pt):  Contradicts any provided field, invents specs, or omits more
    than one of Name/Material/Warranty.

═══ IMPORTANT ═══
- For GROUNDING, compare the description carefully against the SOURCE DATA.
  Flag any detail in the description that is NOT in the source data.
- For LENGTH, count the words in the description yourself.
- Provide your explanation BEFORE your verdict for each criterion.
- Respond ONLY with the requested JSON structure — no extra text.
"""


def build_user_message(row: pd.Series) -> str:
    """Build the user message containing source data + generated description."""
    return f"""\
═══ SOURCE DATA ═══
Product Name: {row['product_name']}
Attributes: {row['Product_attribute_list']}
Material: {row['material']}
Warranty: {row['warranty']}

═══ GENERATED DESCRIPTION ═══
{row['generated_description']}
"""


print("System prompt length:", len(JUDGE_SYSTEM_PROMPT), "chars")
print("Example user message:")
print(build_user_message(df.iloc[0])[:300], "...")

## 5.6 Judge Functions

In [ ]:
def judge_all_criteria(
    row: pd.Series,
    client: OpenAI,
    model: str = JUDGE_MODEL,
) -> dict:
    """Call the judge model once to evaluate all 5 criteria for a single product."""
    user_msg = build_user_message(row)

    t0 = time.time()
    response = client.chat.completions.create(
        model=model,
        temperature=0.0,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": JudgeOutput.model_json_schema(),
        },
    )
    latency_ms = (time.time() - t0) * 1000

    raw = response.choices[0].message.content
    parsed = json.loads(raw)

    result = {}
    for criterion in ["fluency", "grammar", "tone", "length", "grounding"]:
        result[f"{criterion}_explanation"] = parsed[criterion]["explanation"]
        result[f"{criterion}_verdict"] = parsed[criterion]["verdict"]

    result["judge_latency_ms"] = round(latency_ms, 1)
    result["judge_input_tokens"] = response.usage.prompt_tokens
    result["judge_output_tokens"] = response.usage.completion_tokens

    return result


print("judge_all_criteria() defined.")

In [ ]:
# ── Rubric text for single-criterion prompts ───────────────────────────

CRITERION_RUBRICS = {
    "fluency": """\
FLUENCY
  good (3 pts): Smooth, varied sentence structure; flows naturally; no robotic
    repetition; no two sentences start with the same word.
  ok   (2 pts): 1 instance of awkward phrasing, a repeated sentence opener,
    or a robotic transition.
  bad  (1 pt):  3+ awkward phrases, highly repetitive sentence structures,
    or bullet-point style output.""",

    "grammar": """\
GRAMMAR
  good (3 pts): 0 spelling, punctuation, or grammatical errors.
  ok   (2 pts): 1 minor error (e.g., a missing comma) that does not obscure meaning.
  bad  (1 pt):  2+ errors, or any error that makes a sentence difficult to understand.""",

    "tone": """\
TONE
  good (3 pts): Actively persuasive — uses at least two distinct persuasive
    techniques (e.g., benefit framing, emotional/aspirational appeal, sensory
    language, social proof, urgency).
  ok   (2 pts): Uses sales-oriented language but relies on only one persuasive
    technique or describes features without evoking benefit or aspiration
    (informative but flat).
  bad  (1 pt):  Dry/robotic/technical-manual style, overly aggressive, or
    inappropriate.""",

    "length": """\
LENGTH
  good (3 pts): Exactly 55–85 words.
  ok   (2 pts): 50–54 or 86–90 words.
  bad  (1 pt):  Outside the range (<50 or >90 words).""",

    "grounding": """\
GROUNDING
  good (3 pts): Mentions Name, Material, Warranty, AND all structured
    attributes — woven naturally into sentences, with zero hallucinations or
    invented specs.
  ok   (2 pts): Mentions Name, Material, and Warranty but omits one or more
    structured attributes; OR includes 1 minor plausible but unstated detail.
  bad  (1 pt):  Contradicts any provided field, invents specs, or omits more
    than one of Name/Material/Warranty.""",
}


def _single_criterion_system_prompt(criterion_name: str, rubric_text: str) -> str:
    """Build a system prompt focused on a single criterion."""
    grounding_note = ""
    if criterion_name == "grounding":
        grounding_note = (
            "\nIMPORTANT: Compare the description carefully against the "
            "SOURCE DATA. Flag any detail that is NOT in the source data.\n"
        )
    length_note = ""
    if criterion_name == "length":
        length_note = (
            "\nIMPORTANT: Count the words in the description yourself.\n"
        )
    return f"""\
You are an expert product-copy evaluator. You will receive:
1. SOURCE DATA — the original product information (name, attributes, material, warranty).
2. GENERATED DESCRIPTION — a marketing description produced by an LLM.

Evaluate the description on the following criterion ONLY:

{rubric_text}
{grounding_note}{length_note}
First, write a concise explanation of your reasoning.
Then, give your verdict as one of: "good", "ok", or "bad".
Respond ONLY with the requested JSON structure — no extra text.
"""


def judge_single_criterion(
    row: pd.Series,
    criterion_name: str,
    client: OpenAI,
    model: str = JUDGE_MODEL,
) -> dict:
    """Call the judge model to evaluate ONE criterion for a single product."""
    rubric_text = CRITERION_RUBRICS[criterion_name]
    system_prompt = _single_criterion_system_prompt(criterion_name, rubric_text)
    user_msg = build_user_message(row)

    t0 = time.time()
    response = client.chat.completions.create(
        model=model,
        temperature=0.0,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": SingleCriterionResult.model_json_schema(),
        },
    )
    latency_ms = (time.time() - t0) * 1000

    raw = response.choices[0].message.content
    parsed = json.loads(raw)

    return {
        "explanation": parsed["explanation"],
        "verdict": parsed["verdict"],
        "judge_latency_ms": round(latency_ms, 1),
        "judge_input_tokens": response.usage.prompt_tokens,
        "judge_output_tokens": response.usage.completion_tokens,
    }


print("judge_single_criterion() defined.")
print("Criteria available:", list(CRITERION_RUBRICS.keys()))

## 5.7 Pass/Fail Logic

The same rules from Task 1 apply here. Cost and Latency ratings come from
the existing columns in the data (they are measured programmatically, not
by the judge).

In [ ]:
VERDICT_TO_SCORE = {"good": 3, "ok": 2, "bad": 1}
JUDGE_CRITERIA = ["fluency", "grammar", "tone", "length", "grounding"]
ALL_CRITERIA = JUDGE_CRITERIA + ["latency_rating", "cost_rating"]


def compute_pass_fail(row: pd.Series, verdict_suffix: str = "_verdict") -> dict:
    """
    Apply pass/fail logic using judge verdicts + programmatic cost/latency.

    Parameters
    ----------
    row : pd.Series
        Must contain columns like fluency_verdict, ..., grounding_verdict,
        plus latency_rating and cost_rating from the original data.
    verdict_suffix : str
        Suffix for judge verdict columns (default '_verdict').

    Returns
    -------
    dict with total_score, num_good, final_verdict
    """
    # Collect all 7 ratings (5 from judge + 2 programmatic)
    ratings = {}
    for c in JUDGE_CRITERIA:
        ratings[c] = row.get(f"{c}{verdict_suffix}", "bad")
    ratings["latency_rating"] = row.get("latency_rating", "bad")
    ratings["cost_rating"] = row.get("cost_rating", "bad")

    # Compute total score
    total_score = sum(VERDICT_TO_SCORE.get(v, 1) for v in ratings.values())
    num_good = sum(1 for v in ratings.values() if v == "good")

    # Go/No-Go auto-fail checks
    auto_fail = (
        ratings.get("grounding") == "bad"
        or ratings.get("length") == "bad"
        or ratings.get("grammar") == "bad"
    )

    if auto_fail:
        final = "fail"
    elif num_good >= 5 and total_score >= 17:
        final = "pass"
    else:
        final = "fail"

    return {
        "total_score": total_score,
        "num_good": num_good,
        "final_verdict": final,
    }


print("compute_pass_fail() defined.")

---

# Task 6: Run the Judge and Analyze Results

## 6.1 Sanity Check — Judge the First 5 Products

Run the all-criteria judge on a small sample and manually review the
explanations and verdicts to make sure the judge is behaving sensibly.

In [ ]:
# ── Sanity check: judge first 5 products ────────────────────────────────
sanity_results = []

for i in range(min(5, len(df))):
    row = df.iloc[i]
    print(f"[{i+1}/5] Judging: {row['product_name']}")
    try:
        result = judge_all_criteria(row, client)
        result["product_name"] = row["product_name"]
        sanity_results.append(result)
    except Exception as e:
        print(f"  ERROR: {e}")
        sanity_results.append({"product_name": row["product_name"], "error": str(e)})
    time.sleep(0.5)

print("\nDone.")

In [ ]:
# ── Display sanity check results ────────────────────────────────────────
for r in sanity_results:
    if "error" in r:
        print(f"\n{'='*60}")
        print(f"PRODUCT: {r['product_name']}")
        print(f"ERROR: {r['error']}")
        continue

    print(f"\n{'='*60}")
    print(f"PRODUCT: {r['product_name']}")
    print(f"  Latency: {r['judge_latency_ms']:.0f} ms | "
          f"Tokens: {r['judge_input_tokens']}in/{r['judge_output_tokens']}out")
    for c in JUDGE_CRITERIA:
        print(f"\n  [{c.upper()}] verdict={r[f'{c}_verdict']}")
        print(f"    {r[f'{c}_explanation'][:200]}")

## 6.2 Full Run — All Criteria at Once

Run the judge on every product in the dataset. Save results with pass/fail
logic to `task_06_all_criteria.xlsx`.

In [ ]:
# ── Full run: all-criteria-at-once ───────────────────────────────────────
all_criteria_results = []

for idx, row in df.iterrows():
    print(f"[{idx+1}/{len(df)}] Judging: {row['product_name']}")
    try:
        result = judge_all_criteria(row, client)
    except Exception as e:
        print(f"  ERROR: {e}")
        result = {}
        for c in JUDGE_CRITERIA:
            result[f"{c}_explanation"] = f"ERROR: {e}"
            result[f"{c}_verdict"] = "bad"
        result["judge_latency_ms"] = 0
        result["judge_input_tokens"] = 0
        result["judge_output_tokens"] = 0
    all_criteria_results.append(result)
    time.sleep(0.5)

print(f"\nCompleted judging {len(all_criteria_results)} products.")

In [ ]:
# ── Build results DataFrame and apply pass/fail ─────────────────────────
judge_df = df.copy()
results_df = pd.DataFrame(all_criteria_results)

# Add judge columns
for col in results_df.columns:
    judge_df[col] = results_df[col].values

# Apply pass/fail logic
pf_results = judge_df.apply(compute_pass_fail, axis=1, result_type="expand")
judge_df["judge_total_score"] = pf_results["total_score"]
judge_df["judge_num_good"] = pf_results["num_good"]
judge_df["judge_final_verdict"] = pf_results["final_verdict"]

# Summary
print("=== All-Criteria Judge Summary ===")
print(f"Total products: {len(judge_df)}")
print(f"Pass: {(judge_df['judge_final_verdict'] == 'pass').sum()}")
print(f"Fail: {(judge_df['judge_final_verdict'] == 'fail').sum()}")
print(f"\nVerdict distribution per criterion:")
for c in JUDGE_CRITERIA:
    col = f"{c}_verdict"
    counts = judge_df[col].value_counts()
    print(f"  {c:12s}  good={counts.get('good',0):2d}  ok={counts.get('ok',0):2d}  bad={counts.get('bad',0):2d}")

In [ ]:
# ── Save to Excel ────────────────────────────────────────────────────────
output_path = "task_06_all_criteria.xlsx"
judge_df.to_excel(output_path, index=False, engine="openpyxl")
print(f"Saved {len(judge_df)} rows to {output_path}")
print(f"Columns: {judge_df.columns.tolist()}")

## 6.3 Compare Judge to Human Evaluation

Only products that received human evaluation in Task 3 can be compared.
For each of the 5 criteria, compute the agreement rate between the judge
verdict and the human rating.

In [ ]:
def compare_judge_to_human(
    data: pd.DataFrame,
    verdict_suffix: str = "_verdict",
    label: str = "all-at-once",
) -> pd.DataFrame:
    """
    Compare judge verdicts to human ratings for products that have human eval.

    Returns a DataFrame with per-criterion agreement rates.
    """
    # Filter to rows that have human evaluation
    has_human = data["fluency"].notna()
    subset = data[has_human].copy()
    n = len(subset)
    print(f"Comparing {n} products with human evaluation ({label} mode)\n")

    rows = []
    for c in JUDGE_CRITERIA:
        human_col = c  # human ratings are in columns like 'fluency', 'grammar', etc.
        judge_col = f"{c}{verdict_suffix}"

        if judge_col not in subset.columns:
            print(f"  WARNING: {judge_col} not found in data, skipping.")
            continue

        matches = (subset[human_col] == subset[judge_col]).sum()
        agreement = matches / n if n > 0 else 0

        # Detailed breakdown
        details = []
        for _, row in subset.iterrows():
            h = row[human_col]
            j = row[judge_col]
            if h != j:
                details.append(f"{row['product_name'][:25]}: human={h} vs judge={j}")

        rows.append({
            "criterion": c,
            "agreement": f"{agreement:.0%}",
            "matches": matches,
            "total": n,
            "disagreements": "; ".join(details[:5]) if details else "none",
        })

        print(f"  {c:12s}  {matches}/{n} agree ({agreement:.0%})")

    return pd.DataFrame(rows)


# ── Run comparison for all-at-once mode ─────────────────────────────────
agreement_all = compare_judge_to_human(judge_df, "_verdict", "all-at-once")
agreement_all

## 6.4 Criterion-by-Criterion Judging

Instead of evaluating all 5 criteria in one API call, evaluate each criterion
in its own dedicated call. This gives the model a simpler, more focused task
and may improve accuracy — at the cost of 5x more API calls.

In [ ]:
# ── Criterion-by-criterion full run ─────────────────────────────────────
cbc_results = []  # one dict per product

total_calls = len(df) * len(JUDGE_CRITERIA)
call_num = 0

for idx, row in df.iterrows():
    product_result = {}
    total_latency = 0
    total_input_tokens = 0
    total_output_tokens = 0

    for criterion in JUDGE_CRITERIA:
        call_num += 1
        print(f"[{call_num}/{total_calls}] {row['product_name'][:30]} — {criterion}")
        try:
            r = judge_single_criterion(row, criterion, client)
            product_result[f"{criterion}_explanation"] = r["explanation"]
            product_result[f"{criterion}_verdict"] = r["verdict"]
            total_latency += r["judge_latency_ms"]
            total_input_tokens += r["judge_input_tokens"]
            total_output_tokens += r["judge_output_tokens"]
        except Exception as e:
            print(f"  ERROR: {e}")
            product_result[f"{criterion}_explanation"] = f"ERROR: {e}"
            product_result[f"{criterion}_verdict"] = "bad"
        time.sleep(0.5)

    product_result["judge_latency_ms"] = round(total_latency, 1)
    product_result["judge_input_tokens"] = total_input_tokens
    product_result["judge_output_tokens"] = total_output_tokens
    cbc_results.append(product_result)

print(f"\nCompleted criterion-by-criterion judging: {len(cbc_results)} products.")

In [ ]:
# ── Build criterion-by-criterion DataFrame and apply pass/fail ──────────
cbc_df = df.copy()
cbc_results_df = pd.DataFrame(cbc_results)

for col in cbc_results_df.columns:
    cbc_df[col] = cbc_results_df[col].values

# Apply pass/fail
pf_cbc = cbc_df.apply(compute_pass_fail, axis=1, result_type="expand")
cbc_df["judge_total_score"] = pf_cbc["total_score"]
cbc_df["judge_num_good"] = pf_cbc["num_good"]
cbc_df["judge_final_verdict"] = pf_cbc["final_verdict"]

# Save
cbc_output_path = "task_06_product_criteria.xlsx"
cbc_df.to_excel(cbc_output_path, index=False, engine="openpyxl")
print(f"Saved {len(cbc_df)} rows to {cbc_output_path}")

# Summary
print(f"\n=== Criterion-by-Criterion Judge Summary ===")
print(f"Pass: {(cbc_df['judge_final_verdict'] == 'pass').sum()}")
print(f"Fail: {(cbc_df['judge_final_verdict'] == 'fail').sum()}")
print(f"\nVerdict distribution per criterion:")
for c in JUDGE_CRITERIA:
    col = f"{c}_verdict"
    counts = cbc_df[col].value_counts()
    print(f"  {c:12s}  good={counts.get('good',0):2d}  ok={counts.get('ok',0):2d}  bad={counts.get('bad',0):2d}")

## 6.5 Compare Both Judging Modes to Human Evaluation

Side-by-side comparison: did evaluating each criterion in isolation improve
agreement with human ratings?

In [ ]:
# ── Compare criterion-by-criterion to human ─────────────────────────────
print("=" * 60)
print("CRITERION-BY-CRITERION vs HUMAN")
print("=" * 60)
agreement_cbc = compare_judge_to_human(cbc_df, "_verdict", "criterion-by-criterion")

print("\n")
print("=" * 60)
print("ALL-AT-ONCE vs HUMAN (repeated for comparison)")
print("=" * 60)
agreement_all_repeat = compare_judge_to_human(judge_df, "_verdict", "all-at-once")

# ── Side-by-side summary table ──────────────────────────────────────────
print("\n\n=== SIDE-BY-SIDE COMPARISON ===\n")
print(f"{'Criterion':12s}  {'All-at-once':>12s}  {'Crit-by-crit':>12s}")
print("-" * 42)
for c in JUDGE_CRITERIA:
    all_row = agreement_all_repeat[agreement_all_repeat["criterion"] == c]
    cbc_row = agreement_cbc[agreement_cbc["criterion"] == c]
    all_val = all_row["agreement"].values[0] if len(all_row) > 0 else "N/A"
    cbc_val = cbc_row["agreement"].values[0] if len(cbc_row) > 0 else "N/A"
    print(f"  {c:12s}  {all_val:>12s}  {cbc_val:>12s}")

## 6.6 Analysis

### Why might criterion-by-criterion judging produce different results?

When evaluating all criteria at once, the judge's assessment of one criterion can **bleed into another**. For example, if the model notices a grounding issue, it may become more critical of tone or fluency in the same pass. Isolating criteria forces the judge to focus on one dimension at a time, reducing cross-criterion contamination.

Additionally, with a single focused rubric, the smaller model has less instruction to process, which can improve adherence to the specific criterion definitions.

### Trade-offs: Human Evaluation vs LLM-as-Judge

| Dimension | Human Evaluation | LLM-as-Judge |
|---|---|---|
| **Cost** | High (reviewer time) | Low (API calls ~$0.0001 each) |
| **Scale** | 10--15 products feasible | 1000s of products in minutes |
| **Consistency** | Varies with fatigue, mood | Deterministic at temperature=0 |
| **Accuracy** | Gold standard for subjective criteria | Good for clear-cut criteria, weaker on nuance |
| **Grounding** | Humans catch subtle hallucinations | Models may miss plausible-sounding inventions |
| **Speed** | Minutes per product | Seconds per product |

### Recommendation for Production

For a system generating thousands of descriptions daily, use a **tiered approach**:
1. **LLM judge on 100%** of descriptions as a first-pass filter (fast, cheap, consistent)
2. **Human review on failures** and a random sample of passes (~5--10%) to catch judge blind spots
3. **Periodic calibration** -- compare human and judge scores weekly, retune the judge prompt when agreement drops below threshold